## Scenario 3: Multiple data scientists working on multiple ML models

MLflow setup:
* Tracking server: yes, remote server (EC2).
* Backend store: postgresql database.
* Artifacts store: s3 bucket.

The experiments can be explored by accessing the remote server.

The example uses AWS to host a remote server. In order to run the example you'll need an AWS account. Follow the steps described in the file `mlflow_on_aws.md` to create a new AWS account and launch the tracking server. 

In [1]:
import mlflow
import os

os.environ["AWS_PROFILE"] = "default" # fill in with your AWS profile. More info: https://docs.aws.amazon.com/sdk-for-java/latest/developer-guide/setup.html#setup-credentials

TRACKING_SERVER_HOST = "ec2-user@ec2-35-85-32-230.us-west-2.compute.amazonaws.com" # fill in with the public DNS of the EC2 instance
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://ec2-user@ec2-35-85-32-230.us-west-2.compute.amazonaws.com:5000'


In [3]:
mlflow.search_experiments() # list_experiments API has been removed, you can use search_experiments instead.()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-bruke-720881264075-us-west-2-an/1', creation_time=1773525325566, experiment_id='1', last_update_time=1773525325566, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-bruke-720881264075-us-west-2-an/0', creation_time=1773524712704, experiment_id='0', last_update_time=1773524712704, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/03/14 22:05:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 22:05:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
/home/codespace/miniconda3/envs/exp-tracking-env/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


default artifacts URI: 's3://mlflow-artifacts-remote-bruke-720881264075-us-west-2-an/1/cbe3ca298e6b4903bd67b7807c841553/artifacts'
🏃 View run thoughtful-ray-961 at: http://ec2-user@ec2-35-85-32-230.us-west-2.compute.amazonaws.com:5000/#/experiments/1/runs/cbe3ca298e6b4903bd67b7807c841553
🧪 View experiment at: http://ec2-user@ec2-35-85-32-230.us-west-2.compute.amazonaws.com:5000/#/experiments/1


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-bruke-720881264075-us-west-2-an/1', creation_time=1773525325566, experiment_id='1', last_update_time=1773525325566, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-bruke-720881264075-us-west-2-an/0', creation_time=1773524712704, experiment_id='0', last_update_time=1773524712704, lifecycle_stage='active', name='Default', tags={}>]

### Interacting with the model registry

In [6]:
from mlflow.tracking import MlflowClient


client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

In [7]:
client.search_registered_models()

[]

In [8]:
run_id = client.search_runs(experiment_ids=['1'])[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2026/03/14 22:07:54 WARNING mlflow.tracking._model_registry.fluent: Run with id cbe3ca298e6b4903bd67b7807c841553 has no artifacts at artifact path 'models', registering model based on models:/m-c44c7c56dc6e46a8a29918d5df9cac0b instead
2026/03/14 22:07:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1773526074334, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1773526074334, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='cbe3ca298e6b4903bd67b7807c841553', run_link='', source='models:/m-c44c7c56dc6e46a8a29918d5df9cac0b', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [9]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1773526073946, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='', last_updated_timestamp=1773526074334, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1773526074334, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1773526074334, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='cbe3ca298e6b4903bd67b7807c841553', run_link='', source='models:/m-c44c7c56dc6e46a8a29918d5df9cac0b', status='READY', status_message=None, tags={}, user_id='', version='1'>], name='iris-classifier', tags={}>]